# 05 — Frozen Phase Estimator + Phase-Sync Fine-Tuning

**역할**:
1. obs+action에서 phase를 예측하는 frozen **Phase Estimator MLP**를 30 epoch 학습 (supervised circular loss)
2. 04번 trajectory 모델을 `L_total = L_diffusion + 0.12 · L_phase`로 10 epoch fine-tune해 **ours** 체크포인트 생성 (`RUN_PHASE_SYNC_FINE_TUNE=True` 설정)

**산출물**:
- `checkpoints/frozen_phase_estimator_mlp.pt`
- `checkpoints/phase_trajectory_sync_lambda0.12.pt` *(sync fine-tune 활성화 시)*
- `figures/frozen_phase_estimator_training.png`

**예상 소요**: **~20–35분** (Colab T4 GPU; estimator 학습 ~5–10분 + sync fine-tuning ~15–25분)

---

이 노트북은 **실행 orchestration만 담당**합니다. 핵심 로직은 `pcdp/frozen_phase_estimator.py`(`PhaseEstimatorMLP` + `circular_sincos_loss` + `phase_sync_loss` + `train_phase_estimator` + `train_phase_sync_diffusion_policy`), `pcdp/configs.py`(`phase_trajectory_sync` config), `pcdp/experiment_runner.py`(model/scheduler build wrapper), `pcdp/training.py`(`load_checkpoint` / `save_checkpoint` + `trajectory_phase_cond_fn`)에 있습니다.

## 1. Setup

In [ ]:
# Google Drive mount and project-root setup for Colab
from pathlib import Path
import os

PROJECT_DIR = Path('/content/drive/MyDrive/phase_conditioned_diffusion_policy')

try:
    from google.colab import drive
except ImportError:
    print(f'Not running in Google Colab; keeping current working directory: {Path.cwd()}')
else:
    drive.mount('/content/drive')
    if not PROJECT_DIR.exists():
        raise FileNotFoundError(
            f'Expected project directory not found: {PROJECT_DIR}\n'
            'Update PROJECT_DIR to the Google Drive folder that contains this repository.'
        )
    os.chdir(PROJECT_DIR)
    print(f'Current working directory: {Path.cwd()}')

In [ ]:
# Colab dependency setup
import importlib.util
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("google.colab") is None:
    print("Not running in Google Colab; skipping dependency installation and using the current environment.")
else:
    project_root = Path.cwd()
    requirements_path = project_root / "requirements.txt"
    install_command = [sys.executable, "-m", "pip", "install"]

    if requirements_path.exists():
        # requirements.txt pins the runtime stack; -e . installs this repo from pyproject.toml.
        install_command.extend(["-r", str(requirements_path), "-e", str(project_root)])
    else:
        # Fallback to pyproject.toml dependencies if requirements.txt is unavailable.
        install_command.extend(["-e", str(project_root)])

    subprocess.check_call(install_command)

    import gymnasium as gym
    import mujoco
    import minari
    import torch

    gym.make("Ant-v5").close()

    print(f"gymnasium={gym.__version__}")
    print(f"mujoco={mujoco.__version__}")
    print(f"minari={minari.__version__}")
    print(f"torch={torch.__version__}")
    print("Ant-v5 environment smoke check passed.")

## 2. Imports and Paths

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from pcdp.configs import get_experiment_config
from pcdp.dataset import load_project_data, build_loaders
from pcdp.experiment_runner import build_model, build_noise_scheduler
from pcdp.frozen_phase_estimator import (
    PhaseEstimatorConfig,
    build_phase_estimator_from_data,
    evaluate_phase_estimator,
    freeze_phase_estimator,
    load_phase_estimator_checkpoint,
    save_phase_estimator_checkpoint,
    train_phase_estimator,
    train_phase_sync_diffusion_policy,
)
from pcdp.paths import CHECKPOINTS_DIR, DATA_DIR, FIGURES_DIR, ensure_artifact_dirs
from pcdp.training import load_checkpoint, save_checkpoint, trajectory_phase_cond_fn

ensure_artifact_dirs()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}, device={device}')

data_dir = DATA_DIR
print(f'data_dir={data_dir}')
print(f'checkpoints_dir={CHECKPOINTS_DIR}')

## 3. Load Dataset

In [ ]:
data = load_project_data(data_dir)

estimator_cfg = PhaseEstimatorConfig(
    obs_horizon=data['OBS_HORIZON'],
    pred_horizon=data['PRED_HORIZON'],
    obs_dim=data['OBS_DIM'],
    act_dim=data['ACT_DIM'],
    hidden_dim=256,
    num_layers=3,
    dropout=0.05,
    lr=1e-3,
    weight_decay=1e-5,
    num_epochs=30,
)

train_ds, val_ds, train_loader, val_loader = build_loaders(
    data,
    batch_size=256,
    num_workers=2,
)
print(estimator_cfg)

## 4. Train or Load Phase Estimator

In [ ]:
# True: train/overwrite the estimator. False: reuse the existing estimator checkpoint when present.
TRAIN_PHASE_ESTIMATOR = True
estimator_ckpt_path = CHECKPOINTS_DIR / 'frozen_phase_estimator_mlp.pt'

if TRAIN_PHASE_ESTIMATOR or not estimator_ckpt_path.exists():
    estimator = build_phase_estimator_from_data(
        data,
        hidden_dim=estimator_cfg.hidden_dim,
        num_layers=estimator_cfg.num_layers,
        dropout=estimator_cfg.dropout,
        device=device,
    )
    train_losses, val_log, best_state = train_phase_estimator(
        estimator,
        train_loader,
        val_loader,
        device=device,
        num_epochs=estimator_cfg.num_epochs,
        lr=estimator_cfg.lr,
        weight_decay=estimator_cfg.weight_decay,
        grad_clip=estimator_cfg.grad_clip,
        val_n_batches=None,
    )
    save_phase_estimator_checkpoint(
        estimator_ckpt_path,
        estimator,
        config=estimator_cfg,
        train_losses=train_losses,
        val_log=val_log,
        best_state=best_state,
    )
else:
    estimator, meta = load_phase_estimator_checkpoint(estimator_ckpt_path, device=device)
    train_losses = meta.get('train_losses', [])
    val_log = meta.get('val_log', [])

estimator, _ = load_phase_estimator_checkpoint(estimator_ckpt_path, device=device, use_best=True)
freeze_phase_estimator(estimator)
metrics = evaluate_phase_estimator(estimator, val_loader, device=device)
print(metrics)

## 5. Estimator Diagnostics

In [ ]:
if val_log:
    epochs = [int(v['epoch']) for v in val_log]
    val_loss = [v['loss'] for v in val_log]
    val_mae = [v['mae_rad'] for v in val_log]

    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].plot(range(1, len(train_losses) + 1), train_losses, label='train')
    axes[0].plot(epochs, val_loss, label='val')
    axes[0].set_title('Circular loss')
    axes[0].set_xlabel('epoch')
    axes[0].legend()
    axes[1].plot(epochs, val_mae)
    axes[1].set_title('Validation phase MAE')
    axes[1].set_xlabel('epoch')
    axes[1].set_ylabel('radians')
    fig.tight_layout()
    out = FIGURES_DIR / 'frozen_phase_estimator_training.png'
    fig.savefig(out, dpi=160)
    print(f'saved {out}')

## 6. Optional Phase-Sync Fine-Tuning

This section loads the existing `phase_trajectory_ckpt.pt`, freezes the estimator, and fine-tunes the diffusion policy with `L_total = L_diffusion + lambda_phase * L_phase`. Keep it disabled until the estimator validation error is acceptable.

In [ ]:
# True: fine-tune phase_trajectory_ckpt.pt with the frozen estimator loss.
RUN_PHASE_SYNC_FINE_TUNE = False
# Strength of phase synchronization loss; output checkpoint name includes this value.
LAMBDA_PHASE = 0.12

if RUN_PHASE_SYNC_FINE_TUNE:
    cfg = get_experiment_config(
        'phase_trajectory',
        training={
            'num_epochs': 10,
            'lr': 5e-5,
            'weight_decay': 1e-6,
            'val_every': 1,
            'val_n_batches': 8,
            'grad_clip': 1.0,
        },
    )
    model = build_model(cfg, data, device=device)
    ema = cfg.build_ema(model)
    noise_scheduler, _, _ = build_noise_scheduler(cfg)

    base_ckpt_path = CHECKPOINTS_DIR / 'phase_trajectory_ckpt.pt'
    assert base_ckpt_path.exists(), f'missing base checkpoint: {base_ckpt_path}'
    load_checkpoint(base_ckpt_path, model, ema, device=device, use_best_ema=True)

    sync_train_log, sync_val_log, sync_best_ema_state = train_phase_sync_diffusion_policy(
        model,
        ema,
        noise_scheduler,
        estimator,
        train_loader,
        val_loader,
        trajectory_phase_cond_fn,
        device=device,
        num_epochs=cfg.training.num_epochs,
        lr=cfg.training.lr,
        weight_decay=cfg.training.weight_decay,
        lambda_phase=LAMBDA_PHASE,
        phase_warmup_epochs=1,
        val_every=cfg.training.val_every,
        val_n_batches=cfg.training.val_n_batches,
        grad_clip=cfg.training.grad_clip,
    )

    sync_ckpt_path = CHECKPOINTS_DIR / f'phase_trajectory_sync_lambda{LAMBDA_PHASE:g}.pt'
    save_checkpoint(
        sync_ckpt_path,
        model,
        ema,
        sync_train_log,
        sync_val_log,
        sync_best_ema_state,
        config={
            **cfg.to_dict(),
            'phase_estimator_checkpoint': str(estimator_ckpt_path),
            'lambda_phase': LAMBDA_PHASE,
            'phase_warmup_epochs': 1,
        },
    )
    print(f'saved phase-sync checkpoint: {sync_ckpt_path}')